# BirdCLEF 2026 — Perch 2.0 Embedding Precompute
## Standalone TF-only notebook — run this BEFORE the training notebook
## Extracts 1536-dim Google Perch 2.0 embeddings for every training clip
## Saves .npy files to /kaggle/working/ — upload as dataset before training

### Inputs required (add to notebook):
- `birdclef-2026` (competition data)
- `google/bird-vocalization-classifier` → TF2 → perch_v2 → v2

### After this notebook finishes:
1. Go to Kaggle → Datasets → New Dataset
2. Upload /kaggle/working/perch_embeddings/
3. Name it: `birdclef-2026-perch-embs`
4. Add that dataset to birdclef2026-train-v20-perch2.ipynb


In [ ]:
# === CELL 1: INSTALL & IMPORTS ===
import subprocess, sys
# TensorFlow is pre-installed on Kaggle GPU kernels
# Install audio libraries
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "soundfile", "librosa", "tqdm"],
    check=False,
)

import os
import gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import tensorflow as tf
from tqdm.notebook import tqdm

tf.get_logger().setLevel("ERROR")
tf.config.optimizer.set_jit(False)  # Disable XLA — Perch v2 uses XlaCallModule v10 which Kaggle TF doesn't support on GPU; regular CUDA is faster than CPU fallback

print(f"TensorFlow {tf.__version__}")
print(f"GPUs visible to TF: {tf.config.list_physical_devices('GPU')}")

# Perch config — must match training notebook exactly
PERCH_SR      = 32000
SECONDS       = 5
PERCH_TARGET  = PERCH_SR * SECONDS   # 160 000 samples
_PERCH_BATCH  = 32                   # clips per TF forward pass (T4 can handle 32 comfortably)


In [ ]:
# === CELL 2: PATHS ===
def _first_existing(*candidates):
    return next((p for p in candidates if os.path.exists(p)), candidates[0])

TRAIN_META_CSV  = _first_existing(
    "/kaggle/input/birdclef-2026/train.csv",
    "/kaggle/input/competitions/birdclef-2026/train.csv",
)
TRAIN_AUDIO_DIR = _first_existing(
    "/kaggle/input/birdclef-2026/train_audio",
    "/kaggle/input/competitions/birdclef-2026/train_audio",
)
PERCH_MODEL_PATH = _first_existing(
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1",  # CPU version — no XLA, works on GPU too
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/2",
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/perch_v2/1",
    "/kaggle/input/bird-vocalization-classifier/tensorflow2/bird-vocalization-classifier/1",
    "/kaggle/input/bird-vocalization-classifier",
)
EMBD_DIR = Path("/kaggle/working/perch_embeddings")
EMBD_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_META_CSV)

print(f"Training clips  : {len(df)}")
print(f"TRAIN_AUDIO_DIR : {TRAIN_AUDIO_DIR}")
print(f"PERCH_MODEL_PATH: {PERCH_MODEL_PATH}  (exists={os.path.exists(PERCH_MODEL_PATH)})")
print(f"EMBD_DIR        : {EMBD_DIR}")

existing = len(list(EMBD_DIR.glob("*.npy")))
print(f"Already cached  : {existing} embeddings")


In [ ]:
# === CELL 3: EXTRACT PERCH 2.0 EMBEDDINGS ===
assert os.path.exists(PERCH_MODEL_PATH), (
    f"Perch model not found at {PERCH_MODEL_PATH}\n"
    "Add 'google/bird-vocalization-classifier' (TF2 > perch_v2 > v2) to notebook inputs."
)

print(f"Loading Perch 2.0 from {PERCH_MODEL_PATH} ...")
perch_model = tf.saved_model.load(PERCH_MODEL_PATH)
_perch_infer = perch_model.signatures["serving_default"]
print("Model loaded. Starting embedding extraction ...")

_audio_batch, _name_batch = [], []

def _flush_batch():
    audio_tf = tf.constant(np.stack(_audio_batch), dtype=tf.float32)   # (B, 160000)
    out      = _perch_infer(inputs=audio_tf)
    embs     = out["embedding"].numpy()                                  # (B, 1536)
    for name, emb in zip(_name_batch, embs):
        np.save(str(EMBD_DIR / name), emb.astype(np.float32))
    _audio_batch.clear()
    _name_batch.clear()

failed = 0
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Perch embed"):
    audio_path = Path(TRAIN_AUDIO_DIR) / row["filename"]
    emb_name   = row["filename"].replace("/", "_") + ".npy"
    emb_path   = EMBD_DIR / emb_name

    if emb_path.exists():
        skipped += 1
        continue

    try:
        # Read only first 5s worth of samples — avoids loading full 30-60s files
        _info = sf.info(str(audio_path))
        _frames_to_read = int(PERCH_TARGET * (_info.samplerate / PERCH_SR)) + 4096
        y, sr0 = sf.read(str(audio_path), frames=_frames_to_read, always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != PERCH_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=PERCH_SR)
        y = y.astype(np.float32)
        # Pad or trim to exactly SECONDS seconds at PERCH_SR
        if len(y) < PERCH_TARGET:
            y = np.pad(y, (0, PERCH_TARGET - len(y)))
        else:
            y = y[:PERCH_TARGET]
        _audio_batch.append(y)
        _name_batch.append(emb_name)
        if len(_audio_batch) >= _PERCH_BATCH:
            _flush_batch()
    except Exception as e:
        failed += 1

if _audio_batch:
    _flush_batch()

del perch_model, _perch_infer
gc.collect()

total_saved = len(list(EMBD_DIR.glob("*.npy")))
print(f"\nDone!")
print(f"  Saved   : {total_saved}")
print(f"  Skipped : {skipped} (already existed)")
print(f"  Failed  : {failed}")


In [ ]:
# === CELL 4: VERIFY & NEXT STEPS ===
emb_files = list(EMBD_DIR.glob("*.npy"))
print(f"Total embeddings in {EMBD_DIR}: {len(emb_files)}")

# Spot-check one embedding
if emb_files:
    sample = np.load(str(emb_files[0]))
    print(f"Sample shape : {sample.shape}   (expected: (1536,))")
    print(f"Sample dtype : {sample.dtype}")
    print(f"Sample range : [{sample.min():.3f}, {sample.max():.3f}]")
    assert sample.shape == (1536,), f"Unexpected shape: {sample.shape}"
    print("\u2705 Shape check passed")

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("1. Click 'Data' tab on the right -> 'Upload' -> select:")
print(f"   /kaggle/working/perch_embeddings/")
print("2. Create a new Kaggle dataset named: birdclef-2026-perch-embs")
print("3. Add that dataset to birdclef2026-train-v20-perch2.ipynb as input")
print("4. Run the training notebook - no TF needed there, full GPU for PyTorch")
